# Analyse des données de température et d'humidité

In [ ]:
from glob import glob
import numpy as np
import pandas as pd

In [ ]:
fichiers_csv = glob('mesure-impact-projets-verdissement*.csv')
fichiers_csv

* Charger les quatre fichiers de données.

In [ ]:
air_dict_df = {
    nom_fichier: pd.read_csv(nom_fichier)
    for nom_fichier in fichiers_csv
}

for nom_fichier, df in air_dict_df.items():
    print(f'{nom_fichier:60} {len(df):10} lignes')

## Analyse du format des données
### Gestion des colonnes
* Vérifier l'uniformité des noms de colonne.

In [ ]:
def print_apercus():
    for nom_fichier, df in air_dict_df.items():
        print(f'{nom_fichier}:')
        types, valeurs = df.dtypes.to_frame(), df.head(1).transpose()
        types.columns, valeurs.columns = ['Type'], ['Valeur']
        print(pd.concat([types, valeurs], axis='columns'))

print_apercus()

* Fournir le bon séparateur de colonnes pour les deux derniers
  fichiers.

In [ ]:
for nom_fichier in fichiers_csv[-2:]:
    air_dict_df[nom_fichier] = pd.read_csv(nom_fichier, sep=';')

print_apercus()

### Gestion des nombres décimaux
* Fournir en plus le bon séparateur de décimales.

In [ ]:
for nom_fichier in fichiers_csv[-2:]:
    air_dict_df[nom_fichier] = pd.read_csv(
        nom_fichier, sep=';', decimal=',')

print_apercus()

* Corriger le type des `longitude` et `latitude`

In [ ]:
for nom_fichier in fichiers_csv[-2:]:
    air_df = air_dict_df[nom_fichier]
    for nom_colonne in ['longitude', 'latitude']:
        air_df[nom_colonne] = air_df[nom_colonne].astype('float')

print_apercus()

### Gestion des dates et des heures
Puisque les heures sont continues (sans saut ni redondance lors des
changements d'heure), nous supposons que les heures sont en UTC.

* Ajuster le format des dates et heures du premier fichier
  * https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.to_datetime.html

In [ ]:
air_df = air_dict_df[fichiers_csv[0]]
air_df['date_heure'] = pd.to_datetime(air_df['date_heure'])

air_df['date_heure']

* Ajuster le format des dates et heures du deuxième fichier
  * https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes

In [ ]:
air_df = air_dict_df[fichiers_csv[1]]
air_df['date_heure'] = pd.to_datetime(
    air_df['date_heure'],
    format='%m/%d/%y %H:%M:%S'
)

air_df['date_heure']

* Ajuster le format des dates et heures du troisième fichier
  * https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-zone-handling

In [ ]:
air_df = air_dict_df[fichiers_csv[2]]
air_df['date_heure'] = pd.to_datetime(
    air_df['date_heure'],
    format='%m/%d/%y %H:%M:%S %z',
    utc=True,
).dt.tz_convert(None)

air_df['date_heure']

* Ajuster le format des dates et heures du quatrième fichier

In [ ]:
air_df = air_dict_df[fichiers_csv[3]]
air_df['date_heure'] = pd.to_datetime(
    air_df['date_heure'],
    format='%m/%d/%y %H:%M:%S %z',
    utc=True,
).dt.tz_convert(None)

air_df['date_heure']

## Nettoyage de l'ensemble des données
* Regroupons les quatre DataFrame

In [ ]:
air_df = pd.concat(
    [air_dict_df[nom_fichier] for nom_fichier in fichiers_csv]
).reset_index(drop=True)

print(air_df.dtypes)
air_df

### Les stations météo
* Gérer la redondance d'information concernant les stations

In [ ]:
def info_stations():
    stations_df = air_df[
        ['id_station', 'nom_station', 'longitude', 'latitude']
    ].drop_duplicates().sort_values('nom_station')

    print('Colonnes', ' ' * 30, 'Valeurs différentes')
    for nom_col in stations_df.columns:
        print(f'{nom_col:56}', stations_df[nom_col].nunique())
    print(list(stations_df.columns), ' ', len(stations_df))

    return stations_df

info_stations()

* Enlever les `-1` à la fin des identifiants des stations

In [ ]:
air_df['id_station'] = air_df['id_station'].str.replace(
    '-1$', '', regex=True)

info_stations().sort_values('id_station')

* Chercher les lignes avec `S-THC 21317941:21348257` et `Espace_67`

In [ ]:
masque_e67 = (
    (air_df['id_station'] == 'S-THC 21317941:21348257') &
    (air_df['nom_station'] == 'Espace_67')
)

air_df[masque_e67]

* Remplacer `Espace_67` par `Carref_Langelier`

In [ ]:
air_df.loc[masque_e67, 'nom_station'] = 'Carref_Langelier'

info_stations()

* Chercher les lignes avec `Jean-Drapeau_Etang` et la longitude
  `-73.556306`.

In [ ]:
masque_jde = (
    (air_df['nom_station'] == 'Jean-Drapeau_Etang') &
    (air_df['longitude'] == -73.556306)
)

air_df[masque_jde]

* Corriger les coordonnées GPS de certaines stations
  `Jean-Drapeau_Etang`.

In [ ]:
air_df.loc[masque_jde, 'longitude'] = -73.532472
air_df.loc[masque_jde, 'latitude'] = 45.518028

info_stations()

* Enlever les accents dans la colonne `nom_station`.

In [ ]:
air_df['nom_station'] = air_df['nom_station'].str.replace('é', 'e')

info_stations()

* Simplifier l'identifiant des 15 stations à un entier

In [ ]:
id_stations = air_df['id_station']
id_stations = id_stations.str.split().str[1]  # '12345678:87654321'
id_stations = id_stations.str.split(':').str[0]  # '12345678'
id_stations = id_stations.astype('int')  # 12345678

id_stations

* Effectuer le changement d'identifiants dans le DataFrame

In [ ]:
air_df['id_station'] = id_stations
stations_df = info_stations().sort_values('id_station')

stations_df

* Sauvegarder l'information des stations et enlever l'information
  redondante de `air_df`.

In [ ]:
stations_df.to_csv('air_stations.csv', index=False)

for nom_col in stations_df.columns[1:]:
    if nom_col in air_df.columns:
        air_df.drop(columns=nom_col, inplace=True)

temp_rh_dp = air_df.set_index(['id_station', 'date_heure']).copy()
temp_rh_dp

### Les lignes incomplètes
* Enlever les valeurs non définies

In [ ]:
temp_rh_dp = temp_rh_dp.dropna()
temp_rh_dp

### Les températures
* Vérifier s'il y a des valeurs aberrantes

In [ ]:
print(np.sort(temp_rh_dp['temperature'].astype('int').unique()))

* Enlever les valeurs aberrantes

In [ ]:
mediane = temp_rh_dp['temperature'].median()
dev_std = temp_rh_dp['temperature'].std()

temp_rh_dp = temp_rh_dp[
    np.abs(temp_rh_dp['temperature'] - mediane) < 3 * dev_std
]

print(np.sort(temp_rh_dp['temperature'].astype('int').unique()))

### Les valeurs d'humidité relative
* Vérifier s'il y a des valeurs aberrantes

In [ ]:
np.sort(temp_rh_dp['RH'].astype('int').unique())

### Les points de rosée
* Vérifier s'il y a des valeurs aberrantes

In [ ]:
np.sort(temp_rh_dp['dew_point'].astype('int').unique())

* Est-ce que les valeurs de point de rosée sont dépendantes des
  températures et de l'humidité relative?

In [ ]:
points_rosee = temp_rh_dp.copy()

points_rosee['temperature'] = \
    points_rosee['temperature'].round().astype('int')
points_rosee['RH'] = \
    ((points_rosee['RH'] / 5).round() * 5).astype('int')

points_rosee.pivot_table(
    values='dew_point', aggfunc='std',
    index='temperature', columns='RH'
).round(1)

* Sauvegarder une table de conversion
  (température, humidité) -> (point de rosée)

In [ ]:
moyenne_points_rosee = points_rosee.groupby(
    ['temperature', 'RH']
).mean().round(1)

moyenne_points_rosee.to_csv('air_points_rosee.csv')
moyenne_points_rosee.unstack().loc[20:24, :]

* Enlever la colonne `dew_point`

In [ ]:
temp_rh = temp_rh_dp[['temperature', 'RH']].reset_index()
temp_rh

## Agrégation des données
* Extraire différentes composantes de la date et de l'heure.

In [ ]:
temp_rh['date'] = temp_rh['date_heure'].dt.date
temp_rh['heure'] = temp_rh['date_heure'].dt.hour
temp_rh['minute'] = temp_rh['date_heure'].dt.minute

temp_rh

* Quelques statistiques des mesures par station et par date

In [ ]:
temp_rh_par_date = temp_rh.groupby(
    ['id_station', 'date']
)[['temperature', 'RH']].aggregate(
    ['min', 'max', 'mean']
)
temp_rh_par_date

* Renommer les colonnes pour n'avoir qu'un seul niveau.
* Arrondir les moyennes à trois décimales.

In [ ]:
temp_rh_par_date.columns = [
    f'{mesure}_{statistique}'
    for mesure in ['temp', 'rh']
    for statistique in ['min', 'max', 'mean']
]

for mesure in ['temp', 'rh']:
    nom_col = f'{mesure}_mean'
    temp_rh_par_date[nom_col] = temp_rh_par_date[nom_col].round(3)

temp_rh_par_date.to_csv('air_par_date.csv')
pd.read_csv('air_par_date.csv', index_col=['id_station', 'date'])